In [55]:
import pandas as pd
from sqlalchemy import create_engine

In [56]:
# Параметры подключения к локальной БД
db_name = "postgres"
user = "postgres"
password = "123456"
host = "localhost"  
port = "5432"  # Стандартный порт PostgreSQL

In [ ]:
# Создаём подключение через SQLAlchemy
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db_name}") 
engine.execute("SET client_encoding TO 'UTF8';")

In [58]:
# Проверка, что таблица отсортирована по убыванию "Number of Pokemons"
# Ожидаем пустой результат

df = pd.read_sql("""
WITH ordered AS (
    SELECT 
        "Pokemon Type", 
        "Number of Pokemons",
        ROW_NUMBER() OVER (ORDER BY "Number of Pokemons" DESC) AS expected_row,
        ROW_NUMBER() OVER () AS actual_row
    FROM golden.types_statistics
)
SELECT *
FROM ordered
WHERE expected_row != actual_row;
"""
, con=engine)

# Проверка, что DataFrame пустой
assert df.empty, f"EXCEPT вернул непустой результат: \n{df}"

In [52]:
# Проверка корректности суммы количества покемонов с silver.pokemon_types
# Ожидаем равенства значений

df_golden = pd.read_sql("""
SELECT SUM("Number of Pokemons") AS total_pokemons FROM golden.types_statistics
"""
, con=engine)

df_silver = pd.read_sql("""
SELECT COUNT(*) AS total_pokemons FROM silver.pokemon_types
"""
, con=engine)

# Достаём конкретные значения
golden_value = df_golden.iloc[0]['total_pokemons']
silver_value = df_silver.iloc[0]['total_pokemons']

# Проверка равенства значений
assert golden_value == silver_value, f"Сумма покемонов неверна: {golden_value} != {silver_value}"

Pytest - не забудьте установить через pip install pytest

In [54]:
!pytest test_pokemon_types.py

============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.3.5, pluggy-1.5.0
rootdir: /Users/natalia.yakhina/Documents/Documents-lp-0979/DATA QA SQL COURSE/PokemonsETL
plugins: typeguard-4.4.2
collected 2 items                                                              

test_pokemon_types.py ..                                                 [100%]

============================== 2 passed in 0.63s ===============================
